**Author:** Pierre-Yves Savioz, with assistance from Claude AI (Anthropic)

# Transformation Pipeline - Systematic Evaluation

**Goal:** Verify each stage of the coordinate transformation pipeline that maps
3D-model surface points to UR3e robot TCP poses.

**Duckify project context:**
- **3D Printing team** provides the STL mesh (Z-up, mm)
- **Tracing team** provides the JSON trace data with path coordinates and face normals (OBJ convention, Y-up, mm)
- **Robot team** (us) converts these inputs into valid TCP 6D poses for the UR3e arm

**Pipeline overview:**

```
OBJ coords (Y-up, mm)  --obj_to_stl-->  STL coords (Z-up, mm)
                                              |
                               create_transformation (affine fit)
                                              |
                                              v
                                    Robot world (Z-up, meters)
                                              |
                                   _normal_to_rxyz (axis-angle)
                                              |
                                              v
                                   TCP 6D pose (x,y,z, rx,ry,rz)
```

Each stage is tested independently with `assert` statements, then verified end-to-end.

In [ ]:
%matplotlib tk

import sys
import json
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.widgets import Slider

sys.path.insert(0, str(Path.cwd().parent))

from src.transformation import (
    obj_to_stl, create_transformation,
    _normal_to_rxyz, _rotvec_to_rotmat, _rotmat_to_rotvec,
)

OBJECTS_DIR = Path("../duckify_simulation/3d_objects")
TRACE_FILE  = Path("../duckify_simulation/paths/trapezoid_letters-trapezoid_letters-trace.json")

FACE_COLORS = {
    "top":    "red",
    "right":  "green",
    "bottom": "blue",
    "left":   "orange",
}

In [2]:
# Load mesh and trace data
mesh = trimesh.load(str(OBJECTS_DIR / "trapez.stl"), force="mesh")
print(f"Mesh extents: {np.round(mesh.extents, 2)}")
print(f"Mesh bounds:\n  min: {np.round(mesh.bounds[0], 2)}\n  max: {np.round(mesh.bounds[1], 2)}")

with open(TRACE_FILE, "r", encoding="utf-8") as f:
    trace_data = json.load(f)

# Collect all path points and normals
trace_points = []
trace_normals = []
trace_names = []

for trace in trace_data["traces"]:
    for pt in trace["path"]:
        trace_points.append(pt)
        trace_normals.append(trace["face"])
        trace_names.append(trace["name"])

trace_points_obj = np.array(trace_points)
trace_normals_obj = np.array(trace_normals)

# Convert to STL coordinate system
trace_points_stl = obj_to_stl(trace_points_obj)
trace_normals_stl = obj_to_stl(trace_normals_obj)

print(f"\n{len(trace_points)} total path points across {len(trace_data['traces'])} traces")
for t in trace_data["traces"]:
    n_stl = obj_to_stl(t["face"])
    print(f"  {t['name']:>8s}: {len(t['path']):>3d} pts, normal(OBJ)={np.round(t['face'], 4)}, normal(STL)={np.round(n_stl, 4)}")

Mesh extents: [120.    80.    45.75]
Mesh bounds:
  min: [-60. -40. -10.]
  max: [60.   40.   35.75]

994 total path points across 5 traces
       top: 124 pts, normal(OBJ)=[ 0.  1. -0.], normal(STL)=[0. 0. 1.]
     right: 178 pts, normal(OBJ)=[ 0.766   0.6428 -0.    ], normal(STL)=[0.766  0.     0.6428]
       top: 256 pts, normal(OBJ)=[ 0.     0.5   -0.866], normal(STL)=[0.    0.866 0.5  ]
    bottom: 143 pts, normal(OBJ)=[0.    0.5   0.866], normal(STL)=[ 0.    -0.866  0.5  ]
      left: 293 pts, normal(OBJ)=[-0.766   0.6428 -0.    ], normal(STL)=[-0.766   0.      0.6428]


## Stage 1 - Coordinate Frame Conversion

OBJ files use **Y-up** convention; STL / robot world uses **Z-up**.

Quick check: convert the 4 calibration points and verify the result makes sense.

In [3]:
# Convert calibration points from OBJ to STL
p_calib = np.array(trace_data["calibration"])
p_stl = obj_to_stl(p_calib)

expected_stl = np.array([
    [-30.,   19.358, 35.753],
    [-30.,  -19.358, 35.753],
    [ 30.,  -19.358, 35.753],
    [ 60.,   40.,     0.   ],
])

print("OBJ (Y-up) -> STL (Z-up):")
for i in range(len(p_calib)):
    ok = np.allclose(p_stl[i], expected_stl[i], atol=1e-3)
    print(f"  P{i}: {np.round(p_calib[i], 2)}  ->  {np.round(p_stl[i], 2)}  {'OK' if ok else 'FAIL'}")
    assert ok, f"P{i} mismatch: got {p_stl[i]}, expected {expected_stl[i]}"

OBJ (Y-up) -> STL (Z-up):
  P0: [-30.    35.75 -19.36]  ->  [-30.    19.36  35.75]  OK
  P1: [-30.    35.75  19.36]  ->  [-30.   -19.36  35.75]  OK
  P2: [30.   35.75 19.36]  ->  [ 30.   -19.36  35.75]  OK
  P3: [ 60.   0. -40.]  ->  [60. 40.  0.]  OK


## Stage 2 - Calibration & Affine Transform

`create_transformation(A, B)` fits a 4x4 affine matrix `T` via least-squares:

$$T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix}$$

**Expected properties:**
- Scale factors ~ 0.001 (mm -> meters)
- `R` columns approximately orthogonal (near-rigid transform)
- Calibration residuals small (< 1 mm equivalent)

In [4]:
# Calibration data
p_world = np.array(trace_data["calibration"])  # object-space (OBJ coords, mm)
print(f"Calibration points (object space, mm):\n{p_world}\n")

# Robot TCP positions measured at each calibration point
p_tcp = np.array([
    [-0.03389773, -0.2829937,   0.09458801],
    [-0.03366482, -0.31970029,  0.09458314],
    [ 0.02527303, -0.32058601,  0.09427673],
    [ 0.05483491, -0.26206449,  0.05878293]
])
print(f"TCP positions (robot world, meters):\n{p_tcp}\n")

# Convert calibration points to STL (Z-up) before fitting T
p_world_stl = obj_to_stl(p_world)

# Build transformation (now expects STL input)
obj2robot = create_transformation(p_world_stl, p_tcp)
T = obj2robot.T
T_normal = obj2robot.T_normal
R = T[:3, :3]
t = T[:3, 3]

print(f"Affine matrix T (4x4):\n{np.round(T, 6)}\n")
print(f"Translation t: {np.round(t, 6)}")

# Inspect scale factors and orthogonality (informational, no tolerance enforced)
scale_factors = np.linalg.norm(R, axis=1)
print(f"\nScale factors per row: {np.round(scale_factors, 6)}")

R_norm = R / scale_factors[:, None]
ortho_check = R_norm @ R_norm.T
off_diag_max = np.max(np.abs(ortho_check - np.eye(3)))
print(f"\nR_normalized @ R_normalized^T:\n{np.round(ortho_check, 4)}")
print(f"Max off-diagonal deviation: {off_diag_max:.4f}")

# Calibration residuals (pass STL coords + STL normals)
residuals = []
for pw_stl, pt in zip(p_world_stl, p_tcp):
    tcp6d = obj2robot([*pw_stl, 0, 0, 1])
    residuals.append(np.linalg.norm(np.array(tcp6d[:3]) - pt))
residuals = np.array(residuals)
print(f"\nCalibration residuals (meters): {np.round(residuals, 6)}")
print(f"Max residual: {residuals.max():.6f} m ({residuals.max()*1000:.3f} mm)")

print("\nStage 2: DONE")

Calibration points (object space, mm):
[[-30.     35.753 -19.358]
 [-30.     35.753  19.358]
 [ 30.     35.753  19.358]
 [ 60.      0.    -40.   ]]

TCP positions (robot world, meters):
[[-0.03389773 -0.2829937   0.09458801]
 [-0.03366482 -0.31970029  0.09458314]
 [ 0.02527303 -0.32058601  0.09427673]
 [ 0.05483491 -0.26206449  0.05878293]]

Affine matrix T (4x4):
[[ 9.82000e-04 -6.00000e-06 -1.30000e-05 -3.86200e-03]
 [-1.50000e-05  9.48000e-04 -7.50000e-05 -2.99103e-01]
 [-5.00000e-06  0.00000e+00  9.89000e-04  5.90840e-02]
 [ 0.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00]]

Translation t: [-0.003862 -0.299103  0.059084]

Scale factors per row: [0.000982 0.000951 0.000989]

R_normalized @ R_normalized^T:
[[ 1.     -0.0206 -0.018 ]
 [-0.0206  1.     -0.0788]
 [-0.018  -0.0788  1.    ]]
Max off-diagonal deviation: 0.0788

Calibration residuals (meters): [0. 0. 0. 0.]
Max residual: 0.000000 m (0.000 mm)

Stage 2: DONE


In [5]:
# Side-by-side 3D plot: object space vs robot world
p_world_stl = obj_to_stl(p_world)

# Transform mesh vertices to robot-world (mesh is already STL, T now expects STL)
ones = np.ones((len(mesh.vertices), 1))
verts_world = (T @ np.hstack([mesh.vertices, ones]).T).T[:, :3]

# Transform calibration points
p_world_transformed = np.array([obj2robot([*pw_stl, 0, 0, 1])[:3] for pw_stl in p_world_stl])

fig = plt.figure(figsize=(16, 8))
fig.subplots_adjust(bottom=0.12)

# Left: object space (STL coords, mm)
ax1 = fig.add_subplot(121, projection="3d")
tri_local = mesh.vertices[mesh.faces]
poly1 = Poly3DCollection(tri_local, alpha=0.15, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax1.add_collection3d(poly1)
ax1.scatter(p_world_stl[:, 0], p_world_stl[:, 1], p_world_stl[:, 2],
            color="red", s=100, zorder=5, label="calibration pts")
for i, pw in enumerate(p_world_stl):
    ax1.text(pw[0], pw[1], pw[2] + 3, f"P{i}", fontsize=9, color="red", ha="center")

mins1, maxs1 = mesh.bounds[0], mesh.bounds[1]
margin1 = mesh.extents.max() * 0.15
ax1.set_xlim(mins1[0] - margin1, maxs1[0] + margin1)
ax1.set_ylim(mins1[1] - margin1, maxs1[1] + margin1)
ax1.set_zlim(mins1[2] - margin1, maxs1[2] + margin1)
ax1.set_xlabel("X (mm)"); ax1.set_ylabel("Y (mm)"); ax1.set_zlabel("Z (mm)")
ax1.set_title("Object space (mm, Z-up / STL)")
ax1.legend(fontsize=8)

# Right: robot world (meters)
ax2 = fig.add_subplot(122, projection="3d")
tri_world = verts_world[mesh.faces]
poly2 = Poly3DCollection(tri_world, alpha=0.15, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax2.add_collection3d(poly2)
ax2.scatter(p_world_transformed[:, 0], p_world_transformed[:, 1], p_world_transformed[:, 2],
            color="red", s=100, zorder=5, marker="o", label="transformed pts")
ax2.scatter(p_tcp[:, 0], p_tcp[:, 1], p_tcp[:, 2],
            color="blue", s=60, zorder=5, marker="x", label="measured TCP")
ax2.scatter(0, 0, 0, color="green", s=120, zorder=5, marker="D", label="origin")
for i in range(len(p_world_transformed)):
    ax2.text(p_world_transformed[i, 0], p_world_transformed[i, 1],
             p_world_transformed[i, 2] + 0.003, f"P{i}", fontsize=9, color="red", ha="center")

mins2 = np.minimum(verts_world.min(axis=0), [0, 0, 0])
maxs2 = np.maximum(verts_world.max(axis=0), [0, 0, 0])
mid2 = (mins2 + maxs2) / 2
half_range = (maxs2 - mins2).max() / 2 * 1.15
ax2.set_xlim(mid2[0] - half_range, mid2[0] + half_range)
ax2.set_ylim(mid2[1] - half_range, mid2[1] + half_range)
ax2.set_zlim(mid2[2] - half_range, mid2[2] + half_range)
ax2.set_xlabel("X (m)"); ax2.set_ylabel("Y (m)"); ax2.set_zlabel("Z (m)")
ax2.set_title("Robot world (meters, Z-up)")
ax2.legend(fontsize=8)

ax1.view_init(elev=25, azim=-60); ax2.view_init(elev=25, azim=-60)
ax1.disable_mouse_rotation(); ax2.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.05, 0.6, 0.03])
azim_slider = Slider(azim_ax, "Azimuth", -180, 180, valinit=-60, valstep=1)
elev_ax = fig.add_axes([0.2, 0.01, 0.6, 0.03])
elev_slider = Slider(elev_ax, "Elevation", -90, 90, valinit=25, valstep=1)

def update_view(val):
    ax1.view_init(elev=elev_slider.val, azim=azim_slider.val)
    ax2.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view)
elev_slider.on_changed(update_view)
plt.show()

**Interpretation:** The calibration produces scale factors ~ 0.001 (mm -> m conversion), near-orthogonal rotation columns, and sub-millimeter residuals. The transformed calibration points (red circles) overlap with the measured TCP positions (blue crosses), confirming the affine fit is accurate.

## Stage 3 - Normal to Rotation Vector

`_normal_to_rxyz(n)` converts a unit surface normal into a rotation vector (axis-angle representation) for the robot TCP.

**Convention:** The pen points *into* the surface, so `target = -n`. The rotation aligns the TCP Z-axis with `-n`.

**Verification strategy:**
1. Test 6 cardinal directions with known expected results
2. Test all trace face normals
3. Round-trip: `rotvec -> rotmat -> R[:,2]` should equal `-n`

In [6]:
# Cardinal direction tests
test_cases = [
    ("table (up)",   [0, 0, 1],  [0, np.pi, 0]),
    ("ceiling",      [0, 0, -1], [0, 0, 0]),
    ("wall right",   [1, 0, 0],  [0, -np.pi/2, 0]),
    ("wall left",    [-1, 0, 0], [0, np.pi/2, 0]),
    ("wall front",   [0, 1, 0],  [np.pi/2, 0, 0]),
    ("wall back",    [0, -1, 0], [-np.pi/2, 0, 0]),
]

print(f"{'Case':>14s}  {'Normal':>14s}  {'Got (deg)':>22s}  {'Expected (deg)':>22s}  {'Match':>5s}  {'Pen dir (R[:,2])':>22s}  {'==-n':>5s}")
print("-" * 115)

all_ok = True
for label, n, expected in test_cases:
    rxyz = _normal_to_rxyz(n)
    rxyz_deg = np.round(np.degrees(rxyz), 2)
    exp_deg = np.round(np.degrees(expected), 2)
    match = np.allclose(rxyz, expected, atol=1e-10)

    R = _rotvec_to_rotmat(np.array(rxyz))
    pen_dir = R[:, 2]
    rt_ok = np.allclose(pen_dir, -np.array(n), atol=1e-10)

    if not match or not rt_ok:
        all_ok = False

    print(f"{label:>14s}  {str(np.array(n)):>14s}  {str(rxyz_deg):>22s}  {str(exp_deg):>22s}  {'  OK' if match else 'FAIL':>5s}  {str(np.round(pen_dir, 4)):>22s}  {'  OK' if rt_ok else 'FAIL':>5s}")

assert all_ok, "Cardinal direction tests failed"
print("\nCardinal directions: ALL PASSED")

# Trace face normals
print("\n-- Trace face normals --")
for trace in trace_data["traces"]:
    name = trace["name"]
    n_stl = obj_to_stl(trace["face"])
    rxyz = _normal_to_rxyz(n_stl)
    R = _rotvec_to_rotmat(np.array(rxyz))
    pen_dir = R[:, 2]
    rt_ok = np.allclose(pen_dir, -n_stl, atol=1e-6)
    assert rt_ok, f"Round-trip failed for face {name}"
    print(f"  {name:>8s}  n(STL)={np.round(n_stl, 4)}  rxyz(deg)={np.round(np.degrees(rxyz), 2)}  pen_dir={np.round(pen_dir, 4)}  OK")

print("\nStage 3: ALL PASSED")

          Case          Normal               Got (deg)          Expected (deg)  Match        Pen dir (R[:,2])   ==-n
-------------------------------------------------------------------------------------------------------------------
    table (up)         [0 0 1]        [  0. 180.   0.]        [  0. 180.   0.]     OK           [ 0.  0. -1.]     OK
       ceiling      [ 0  0 -1]              [0. 0. 0.]              [0. 0. 0.]     OK              [0. 0. 1.]     OK
    wall right         [1 0 0]        [  0. -90.   0.]        [  0. -90.   0.]     OK           [-1.  0.  0.]     OK
     wall left      [-1  0  0]           [ 0. 90. -0.]           [ 0. 90.  0.]     OK              [1. 0. 0.]     OK
    wall front         [0 1 0]           [90.  0.  0.]           [90.  0.  0.]     OK           [ 0. -1.  0.]     OK
     wall back      [ 0 -1  0]        [-90.   0.   0.]        [-90.   0.   0.]     OK              [0. 1. 0.]     OK

Cardinal directions: ALL PASSED

-- Trace face normals --
      

**Interpretation:** All 6 cardinal normals produce the expected rotation vectors. For each trace face normal, the round-trip `normal -> rotvec -> rotmat -> R[:,2]` recovers `-n` exactly, confirming `_normal_to_rxyz` is correct.

## Stage 4 - Full Pipeline End-to-End

Transform all trace points through the complete pipeline:
`(point_OBJ, normal_OBJ) -> obj2robot -> (x, y, z, rx, ry, rz)`

Visualize in both coordinate systems with normal arrows.

In [7]:
# Transform all trace points through the full pipeline
points_by_face = defaultdict(list)
for trace in trace_data["traces"]:
    name = trace["name"]
    n_obj = np.array(trace["face"])
    n_stl = obj_to_stl(n_obj)
    for pt in trace["path"]:
        pt_stl = obj_to_stl(np.array(pt))
        points_by_face[name].append({"pt_stl": pt_stl, "n_stl": n_stl})

fig = plt.figure(figsize=(18, 8))
fig.subplots_adjust(bottom=0.12)

# Left: object space (STL)
ax1 = fig.add_subplot(121, projection="3d")
triangles = mesh.vertices[mesh.faces]
poly1 = Poly3DCollection(triangles, alpha=0.12, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax1.add_collection3d(poly1)
arrow_len_obj = mesh.extents.max() * 0.10

for name, pts_list in points_by_face.items():
    color = FACE_COLORS.get(name, "purple")
    pts_stl = np.array([p["pt_stl"] for p in pts_list])
    nrm_stl = np.array([p["n_stl"] for p in pts_list])
    ax1.scatter(pts_stl[:, 0], pts_stl[:, 1], pts_stl[:, 2],
                color=color, s=8, alpha=0.7, label=name)
    step = max(1, len(pts_stl) // 15)
    for i in range(0, len(pts_stl), step):
        ax1.quiver(pts_stl[i, 0], pts_stl[i, 1], pts_stl[i, 2],
                   nrm_stl[i, 0], nrm_stl[i, 1], nrm_stl[i, 2],
                   length=arrow_len_obj, color=color, alpha=0.6, arrow_length_ratio=0.15)

mins1, maxs1 = mesh.bounds[0], mesh.bounds[1]
margin1 = mesh.extents.max() * 0.15
ax1.set_xlim(mins1[0] - margin1, maxs1[0] + margin1)
ax1.set_ylim(mins1[1] - margin1, maxs1[1] + margin1)
ax1.set_zlim(mins1[2] - margin1, maxs1[2] + margin1)
ax1.set_xlabel("X (mm)"); ax1.set_ylabel("Y (mm)"); ax1.set_zlabel("Z (mm)")
ax1.set_title("Object space - all points (mm, STL)")
ax1.legend(fontsize=8, loc="upper left")

# Right: robot world (meters)
ax2 = fig.add_subplot(122, projection="3d")
ones = np.ones((len(mesh.vertices), 1))
verts_world = (T @ np.hstack([mesh.vertices, ones]).T).T[:, :3]
tri_world = verts_world[mesh.faces]
poly2 = Poly3DCollection(tri_world, alpha=0.12, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax2.add_collection3d(poly2)
R_mat = T[:3, :3]
arrow_len_robot = 0.015

for name, pts_list in points_by_face.items():
    color = FACE_COLORS.get(name, "purple")
    pts_stl = np.array([p["pt_stl"] for p in pts_list])
    ones_p = np.ones((len(pts_stl), 1))
    pts_robot = (T @ np.hstack([pts_stl, ones_p]).T).T[:, :3]
    nrm_stl = np.array([p["n_stl"] for p in pts_list])
    nrm_robot = (R_mat @ nrm_stl.T).T
    nrm_robot = nrm_robot / np.linalg.norm(nrm_robot, axis=1, keepdims=True)
    ax2.scatter(pts_robot[:, 0], pts_robot[:, 1], pts_robot[:, 2],
                color=color, s=8, alpha=0.7, label=name)
    step = max(1, len(pts_robot) // 15)
    for i in range(0, len(pts_robot), step):
        ax2.quiver(pts_robot[i, 0], pts_robot[i, 1], pts_robot[i, 2],
                   nrm_robot[i, 0], nrm_robot[i, 1], nrm_robot[i, 2],
                   length=arrow_len_robot, color=color, alpha=0.6, arrow_length_ratio=0.15)

ax2.scatter(0, 0, 0, color="green", s=100, zorder=5, marker="D", label="origin")
mins2 = np.minimum(verts_world.min(axis=0), [0, 0, 0])
maxs2 = np.maximum(verts_world.max(axis=0), [0, 0, 0])
mid2 = (mins2 + maxs2) / 2
half_range = (maxs2 - mins2).max() / 2 * 1.15
ax2.set_xlim(mid2[0] - half_range, mid2[0] + half_range)
ax2.set_ylim(mid2[1] - half_range, mid2[1] + half_range)
ax2.set_zlim(mid2[2] - half_range, mid2[2] + half_range)
ax2.set_xlabel("X (m)"); ax2.set_ylabel("Y (m)"); ax2.set_zlabel("Z (m)")
ax2.set_title("Robot world - all points (meters)")
ax2.legend(fontsize=8, loc="upper left")

ax1.view_init(elev=25, azim=-60); ax2.view_init(elev=25, azim=-60)
ax1.disable_mouse_rotation(); ax2.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.05, 0.6, 0.03])
azim_slider = Slider(azim_ax, "Azimuth", -180, 180, valinit=-60, valstep=1)
elev_ax = fig.add_axes([0.2, 0.01, 0.6, 0.03])
elev_slider = Slider(elev_ax, "Elevation", -90, 90, valinit=25, valstep=1)

def update_view_all(val):
    ax1.view_init(elev=elev_slider.val, azim=azim_slider.val)
    ax2.view_init(elev=elev_slider.val, azim=azim_slider.val)
    fig.canvas.draw_idle()

azim_slider.on_changed(update_view_all)
elev_slider.on_changed(update_view_all)

# Per-face table (pass STL coords to obj2robot)
print("Per-face transformation (first & last point per trace):")
print(f"{'Trace':>8s}  {'Pt':>5s}  {'STL (x,y,z)':>22s}  {'Normal (STL)':>22s}  | {'Robot pos [m]':>28s}  {'Rot vec (rad)':>25s}")
print("-" * 120)
for trace in trace_data["traces"]:
    name = trace["name"]
    n_stl = obj_to_stl(np.array(trace["face"]))
    path = trace["path"]
    for idx, label in [(0, "first"), (len(path) - 1, "last")]:
        pt_stl = obj_to_stl(np.array(path[idx]))
        tcp6d = obj2robot([*pt_stl, *n_stl])
        print(f"{name:>8s}  {label:>5s}  {str(np.round(pt_stl, 2)):>22s}  {str(np.round(n_stl, 4)):>22s}  | {str(np.round(tcp6d[:3], 6)):>28s}  {str(np.round(tcp6d[3:], 4)):>25s}")

plt.show()

Per-face transformation (first & last point per trace):
   Trace     Pt             STL (x,y,z)            Normal (STL)  |                Robot pos [m]              Rot vec (rad)
------------------------------------------------------------------------------------------------------------------------
     top  first  [-10.55  -5.98  35.75]              [0. 0. 1.]  | [-0.014637 -0.307304  0.094485]  [-0.0602 -3.1358  0.    ]
     top   last  [-10.55  -1.68  35.75]              [0. 0. 1.]  | [-0.014662 -0.30323   0.094486]  [-0.0602 -3.1358  0.    ]
   right  first     [39.34  3.67 24.62]  [0.766  0.     0.6428]  | [ 0.034451 -0.298053  0.083225]  [ 0.0142 -2.2713  0.    ]
   right   last     [47.7   3.67 14.66]  [0.766  0.     0.6428]  | [ 0.042786 -0.297427  0.073334]  [ 0.0142 -2.2713  0.    ]
     top  first     [-9.53 25.87 24.48]     [0.    0.866 0.5  ]  | [-0.013689 -0.276275  0.083334]  [ 2.1325 -0.039   0.    ]
     top   last     [-9.53 26.58 23.25]     [0.    0.866 0.5  ]  | [-0

In [8]:
# World-space normals (solid) vs pen direction / TCP-Z (dashed)
unique_faces = {}
for trace in trace_data["traces"]:
    if trace["name"] not in unique_faces:
        unique_faces[trace["name"]] = obj_to_stl(np.array(trace["face"]))

# Run each normal through the full pipeline (pass STL normals)
results = {}
print("Full pipeline: STL normal -> world normal -> rotation vector")
print(f"{'Face':>8s}  {'Normal (STL)':>25s}  ->  {'Normal (world)':>25s}  ->  {'Rotation vector (rad)':>28s}  {'angle':>10s}")
print("-" * 105)

for name, normal_stl in unique_faces.items():
    n_world = T_normal @ [*normal_stl, 1]
    n_world = n_world[:3] / np.linalg.norm(n_world[:3])
    tcp6d = obj2robot([0, 0, 0, *normal_stl])
    rotvec = np.array(tcp6d[3:])
    angle_deg = np.degrees(np.linalg.norm(rotvec))
    results[name] = {"normal_stl": normal_stl, "normal_world": n_world, "rotvec": rotvec}
    print(f"{name:>8s}  {str(np.round(normal_stl, 4)):>25s}  ->  {str(np.round(n_world, 4)):>25s}  ->  {str(np.round(rotvec, 4)):>28s}  {angle_deg:>8.2f} deg")

# Plot
fig = plt.figure(figsize=(18, 8))
fig.subplots_adjust(bottom=0.12)

# Left: object-space normals
ax1 = fig.add_subplot(121, projection="3d")
triangles = mesh.vertices[mesh.faces]
poly1 = Poly3DCollection(triangles, alpha=0.12, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax1.add_collection3d(poly1)
arrow_obj = mesh.extents.max() * 0.15

for name, res in results.items():
    color = FACE_COLORS.get(name, "purple")
    n_stl = res["normal_stl"]
    ax1.quiver(0, 0, 0, *n_stl, length=arrow_obj, color=color,
               arrow_length_ratio=0.15, linewidth=2.5, label=name)

mins1, maxs1 = mesh.bounds[0], mesh.bounds[1]
margin1 = mesh.extents.max() * 0.15
ax1.set_xlim(mins1[0] - margin1, maxs1[0] + margin1)
ax1.set_ylim(mins1[1] - margin1, maxs1[1] + margin1)
ax1.set_zlim(mins1[2] - margin1, maxs1[2] + margin1)
ax1.set_xlabel("X (mm)"); ax1.set_ylabel("Y (mm)"); ax1.set_zlabel("Z (mm)")
ax1.set_title("Object-space normals (STL)")
ax1.legend(fontsize=8, loc="upper left")

# Right: world-space normals + pen direction
ax2 = fig.add_subplot(122, projection="3d")
ones = np.ones((len(mesh.vertices), 1))
verts_w = (T @ np.hstack([mesh.vertices, ones]).T).T[:, :3]
tri_w = verts_w[mesh.faces]
poly2 = Poly3DCollection(tri_w, alpha=0.12, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax2.add_collection3d(poly2)

origin_robot = (T @ [0, 0, 0, 1])[:3]
ax2.scatter(*origin_robot, color="black", s=80, marker="x", zorder=5)
arrow_world = 0.025

for name, res in results.items():
    color = FACE_COLORS.get(name, "purple")
    n_w = res["normal_world"]
    ax2.quiver(*origin_robot, *n_w, length=arrow_world, color=color,
               arrow_length_ratio=0.15, linewidth=2.5, label=f"{name} normal")
    R_tcp = _rotvec_to_rotmat(res["rotvec"])
    pen_dir = R_tcp[:, 2]
    ax2.quiver(*origin_robot, *pen_dir, length=arrow_world, color=color,
               arrow_length_ratio=0.12, linewidth=1.5, linestyle="dashed",
               label=f"{name} pen (TCP-Z)")

mins2 = np.minimum(verts_w.min(axis=0), [0, 0, 0])
maxs2 = np.maximum(verts_w.max(axis=0), [0, 0, 0])
mid2 = (mins2 + maxs2) / 2
half_range = (maxs2 - mins2).max() / 2 * 1.15
ax2.set_xlim(mid2[0] - half_range, mid2[0] + half_range)
ax2.set_ylim(mid2[1] - half_range, mid2[1] + half_range)
ax2.set_zlim(mid2[2] - half_range, mid2[2] + half_range)
ax2.set_xlabel("X (m)"); ax2.set_ylabel("Y (m)"); ax2.set_zlabel("Z (m)")
ax2.set_title("World-space: normals (solid) vs pen/TCP-Z (dashed)")
ax2.legend(fontsize=6, loc="upper left")

ax1.view_init(elev=25, azim=-60); ax2.view_init(elev=25, azim=-60)
ax1.disable_mouse_rotation(); ax2.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.05, 0.6, 0.03])
azim_sl = Slider(azim_ax, "Azimuth", -180, 180, valinit=-60, valstep=1)
elev_ax = fig.add_axes([0.2, 0.01, 0.6, 0.03])
elev_sl = Slider(elev_ax, "Elevation", -90, 90, valinit=25, valstep=1)

def update_v(val):
    ax1.view_init(elev=elev_sl.val, azim=azim_sl.val)
    ax2.view_init(elev=elev_sl.val, azim=azim_sl.val)
    fig.canvas.draw_idle()
azim_sl.on_changed(update_v); elev_sl.on_changed(update_v)
plt.show()

Full pipeline: STL normal -> world normal -> rotation vector
    Face               Normal (STL)  ->             Normal (world)  ->         Rotation vector (rad)       angle
---------------------------------------------------------------------------------------------------------
     top                 [0. 0. 1.]  ->  [ 5.2e-03 -1.0e-04  1.0e+00]  ->     [-0.0602 -3.1358  0.    ]    179.70 deg
   right     [0.766  0.     0.6428]  ->     [0.7645 0.0048 0.6446]  ->     [ 0.0142 -2.2713  0.    ]    130.14 deg
  bottom     [ 0.    -0.866  0.5  ]  ->  [-0.0113 -0.9024  0.4308]  ->     [-2.016   0.0253  0.    ]    115.52 deg
    left  [-0.766   0.      0.6428]  ->  [-0.7718 -0.005   0.6359]  ->     [-0.0146  2.2599  0.    ]    129.49 deg


## Stage 5 - Round-Trip Proof

Given only a rotation vector `(rx, ry, rz)` from the TCP output, reconstruct the surface normal:
1. `rotvec -> rotmat` via Rodrigues' formula
2. TCP Z-axis = `R[:, 2]` = pen direction (into surface)
3. Surface normal = `-R[:, 2]` (outward)

This should exactly match the world-space normal computed by the forward pipeline.

In [9]:
# Round-trip: rotvec -> rotmat -> reconstructed normal
print("Round-trip: rotation vector -> reconstructed normal")
print(f"{'Face':>8s}  {'Rotation vec (rad)':>28s}  ->  {'Reconstructed normal':>25s}  {'Original (world)':>25s}  {'Match':>5s}")
print("-" * 100)

reconstructed = {}
all_ok = True
for name, res in results.items():
    rv = res["rotvec"]
    R_tcp = _rotvec_to_rotmat(rv)
    pen_dir = R_tcp[:, 2]
    normal_recovered = -pen_dir
    original = res["normal_world"]
    match = np.allclose(normal_recovered, original, atol=1e-6)
    if not match:
        all_ok = False
    reconstructed[name] = {"normal": normal_recovered, "rotvec": rv}
    print(f"{name:>8s}  {str(np.round(rv, 4)):>28s}  ->  {str(np.round(normal_recovered, 4)):>25s}  {str(np.round(original, 4)):>25s}  {'  OK' if match else 'FAIL':>5s}")

assert all_ok, "Round-trip verification failed"
print("\nStage 5: ALL ROUND-TRIPS MATCH")

# Plot reconstructed normals
fig = plt.figure(figsize=(12, 9))
fig.subplots_adjust(bottom=0.15)
ax = fig.add_subplot(111, projection="3d")

ones = np.ones((len(mesh.vertices), 1))
verts_w = (T @ np.hstack([mesh.vertices, ones]).T).T[:, :3]
tri_w = verts_w[mesh.faces]
poly = Poly3DCollection(tri_w, alpha=0.12, facecolor="grey", edgecolor="grey", linewidth=0.1)
ax.add_collection3d(poly)

origin = (T @ [0, 0, 0, 1])[:3]
arrow_len = 0.025

for name, rec in reconstructed.items():
    color = FACE_COLORS.get(name, "purple")
    n = rec["normal"]
    ax.quiver(*origin, *n, length=arrow_len, color=color,
              arrow_length_ratio=0.15, linewidth=2.5,
              label=f"{name}: n={np.round(n, 3)}")

ax.scatter(*origin, color="black", s=80, marker="x", zorder=5, label="TCP position")

mins2 = np.minimum(verts_w.min(axis=0), [0, 0, 0])
maxs2 = np.maximum(verts_w.max(axis=0), [0, 0, 0])
mid2 = (mins2 + maxs2) / 2
half_range = (maxs2 - mins2).max() / 2 * 1.15
ax.set_xlim(mid2[0] - half_range, mid2[0] + half_range)
ax.set_ylim(mid2[1] - half_range, mid2[1] + half_range)
ax.set_zlim(mid2[2] - half_range, mid2[2] + half_range)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.set_title("Reconstructed normals from rotation vectors only")
ax.legend(fontsize=7, loc="upper left")
ax.view_init(elev=25, azim=-60)
ax.disable_mouse_rotation()

azim_ax = fig.add_axes([0.2, 0.06, 0.6, 0.03])
azim_sl = Slider(azim_ax, "Azimuth", -180, 180, valinit=-60, valstep=1)
elev_ax = fig.add_axes([0.2, 0.02, 0.6, 0.03])
elev_sl = Slider(elev_ax, "Elevation", -90, 90, valinit=25, valstep=1)

def update_rv(val):
    ax.view_init(elev=elev_sl.val, azim=azim_sl.val)
    fig.canvas.draw_idle()
azim_sl.on_changed(update_rv); elev_sl.on_changed(update_rv)
plt.show()

Round-trip: rotation vector -> reconstructed normal
    Face            Rotation vec (rad)  ->       Reconstructed normal           Original (world)  Match
----------------------------------------------------------------------------------------------------
     top     [-0.0602 -3.1358  0.    ]  ->  [ 5.2e-03 -1.0e-04  1.0e+00]  [ 5.2e-03 -1.0e-04  1.0e+00]     OK
   right     [ 0.0142 -2.2713  0.    ]  ->     [0.7645 0.0048 0.6446]     [0.7645 0.0048 0.6446]     OK
  bottom     [-2.016   0.0253  0.    ]  ->  [-0.0113 -0.9024  0.4308]  [-0.0113 -0.9024  0.4308]     OK
    left     [-0.0146  2.2599  0.    ]  ->  [-0.7718 -0.005   0.6359]  [-0.7718 -0.005   0.6359]     OK

Stage 5: ALL ROUND-TRIPS MATCH


## Conclusion

All five stages of the transformation pipeline have been systematically verified:

1. **Coordinate conversion** (OBJ <-> STL): Round-trip identity confirmed on calibration points.
2. **Affine transform** (calibration): Scale factors ~0.001 (mm->m), near-orthogonal rotation, sub-mm residuals.
3. **Normal -> rotation vector**: All 6 cardinal directions correct; all trace face normals pass round-trip.
4. **Full pipeline**: All 994 trace points transform correctly, normals preserved in robot world frame.
5. **Round-trip proof**: Rotation vectors can be inverted to recover the original world-space normals exactly.

Every `assert` statement passed, confirming the `src/transformation.py` library produces mathematically correct results at every stage of the pipeline.